# Fault-Tolerance Engine: End-to-End Tutorial

This advanced tutorial walks a circuit through the full Classiq **Fault-Tolerance (FT) Engine** pipeline, end to end:

1. **Get a logical noise model** (a ready-made public one, or initialize your own).
2. **Model** a quantum program in Qmod (a small Quantum Phase Estimation).
3. **Synthesize** it to a quantum program.
4. **Export** it to a Clifford+T circuit.
5. **Route** the circuit onto a 3-D surface-code lattice.
6. **Visualize** the routed program.
7. Summarize **resources, fidelity, and runtime** across code distances.

> The FT-engine steps (logical-noise initialization, routing, visualization, and total-error estimation) run on the Classiq backend and require an account with FT-engine access. For a conceptual overview see the [Error Correction user guide](https://docs.classiq.io/user-guide/error-correction); for the full API surface see the [SDK reference](https://docs.classiq.io/sdk-reference/error-correction).

## Setup

In [1]:
!pip install -U classiq


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import time

import pandas as pd

from classiq import *
from classiq.error_correction.logical_noise import (
    get_logical_noise,
    initialize_logical_noise,
    remove_logical_noise,
)
from classiq.error_correction.physical_noise import (
    DepolarizeRule,
    PhysicalNoiseModel,
)
from classiq.error_correction.routing import route_circuit
from classiq.error_correction.total_error import estimate_total_errors
from classiq.error_correction.viewer import view_program

/Users/ron/Projects/classiq-library/.venv/lib/python3.11/site-packages/classiq/__init__.py:109: UserWarning: 'nest_asyncio2' is not installed; falling back to the legacy 'nest_asyncio'. 'nest_asyncio2' ships with modern Jupyter (ipykernel >= 7.3); consider upgrading your Jupyter installation, for example: pip install --upgrade jupyter ipykernel
  enable_jupyter_notebook()


ModuleNotFoundError: No module named 'classiq.error_correction.physical_noise'

## 1. Get a logical noise model

On error-corrected hardware each logical qubit is a patch of many physical qubits protected by a surface code. A **logical noise model** captures how the logical error rate of each fundamental operation shrinks as the code distance grows, given the hardware's physical noise.

There are two ways to get one:

- **Use a ready-made public model** (what we do here) — retrieve it by name with `get_logical_noise`, no simulation required.
- **Initialize your own** from a `PhysicalNoiseModel` with `initialize_logical_noise`. This runs a Monte-Carlo simulation on the backend and can take up to ~1 hour, so it is shown below but left off by default. Flip `INITIALIZE_OWN_NOISE` to `True` to run it once (the result is stored under your chosen name and reused afterwards).

See the [Logical Noise user guide](https://docs.classiq.io/user-guide/error-correction/logical-noise) for more detail on public models, custom noise, and plotting.

In [ ]:
# Initializing your own logical noise is a long backend simulation (~1 hour), so it is off
# by default; the tutorial uses the shared public model instead.
INITIALIZE_OWN_NOISE = False

if INITIALIZE_OWN_NOISE:
    physical_noise = PhysicalNoiseModel(
        idle=[DepolarizeRule(p=1e-4)],
        clifford_1q=[DepolarizeRule(p=1e-4)],
        clifford_2q=[DepolarizeRule(p=1e-3)],
        measure={"Z": 5e-3, "X": 5e-3},
    )
    NOISE_NAME = "ft_tutorial_noise"
    initialize_logical_noise(NOISE_NAME, physical_noise)
else:
    # A general-purpose depolarizing model published for everyone -- no simulation to run.
    NOISE_NAME = "public/depolarize_1e-3"

logical_noise = get_logical_noise(NOISE_NAME)

The retrieved `logical_noise` exposes the fitted logical error rates. For example, plot the 1-qubit Clifford logical error as a function of the code distance — it drops exponentially, which is the whole point of error correction.

In [ ]:
logical_noise.plot_clifford(start=5, end=25)

The 1-qubit Clifford logical error is essentially a **memory experiment** — it measures how well the surface code preserves a logical qubit over one round of stabilizer measurements under the given noise model. The exponential suppression with code distance is the hallmark of a well-below-threshold error-correcting code.

In [ ]:
logical_noise.plot_cnot(start=5, end=25)

A CNOT in the surface code is performed via **lattice surgery** — merging and splitting patches along a bus of ancilla patches. Of the six Pauli error channels (IX, XI, XX, IZ, ZI, ZZ), two — **IX** and **ZI** — have correlation surfaces that span the entire bus. Their error rate therefore grows with the number of patches between control and target: the longer the CNOT, the more physical qubits participate, and the more opportunities for error. The other four channels are bounded by the fixed-size towers at each endpoint and do not scale with bus length.

In [ ]:
logical_noise.plot_s(start=5, end=25)

## 2. Build the circuit in Qmod

Any circuit works; here we use a small **Quantum Phase Estimation** that reads the phase of a `T` gate (eigenphase `1/8`) into a 3-qubit register. QPE is a good FT example because it is naturally expressed in the Clifford+T gate set the surface code implements.

In [ ]:
PHASE_BITS = 3


@qfunc
def main(
    estimated_phase: Output[QNum[PHASE_BITS, UNSIGNED, PHASE_BITS]],
    target: Output[QBit],
) -> None:
    allocate(PHASE_BITS, estimated_phase)
    allocate(target)
    X(target)  # |1> is an eigenstate of T
    qpe(unitary=lambda: T(target), phase=estimated_phase)

## 3. Synthesize

In [ ]:
qprog = synthesize(main)
show(qprog)

> **Tip:** You can also feed an externally prepared QASM string directly into the FT engine — see [`qasm_to_qmod`](https://docs.classiq.io/sdk-reference/synthesis#qasm_to_qmod) in the SDK reference.

## 4. Export to a Clifford+T circuit

Routing expects a transpiled **Clifford+T** circuit. `export` with a `FaultTolerantTranspilationConfig` decomposes the program into Clifford+T, approximating any rotations to within `clifford_t_approximation_error`.

In [ ]:
ft_config = FaultTolerantTranspilationConfig(clifford_t_approximation_error=1e-2)

ft_qasm = export(
    qprog,
    target_language=TargetLanguage.QASM2,
    transpilation_config=ft_config,
)

## 5. Route onto the surface-code lattice

`route_circuit` lowers the Clifford+T circuit onto a 3-D surface-code lattice — two spatial axes for the grid of surface-code patches and a third (z) axis for time, in error-correction cycles. It returns a `TopologicalProgram` describing where and when every operation happens. See the [Routing user guide](https://docs.classiq.io/user-guide/error-correction/routing) for more information.

In [ ]:
route_start = time.perf_counter()
topo_program = route_circuit(ft_qasm)
route_seconds = time.perf_counter() - route_start

In [ ]:
stats = topo_program.stats
layout = topo_program.layout
num_patches = layout.width * layout.height
print(
    f"Logical qubits: {stats.qubit_count}   |   cycles: {stats.cycles}   |   "
    f"patch grid: {layout.width}x{layout.height} ({num_patches} patches)"
)
print(f"Runtime  ->  routing: {route_seconds:.1f} s")

## 6. Visualize the routed program

`view_program` renders the routed lattice as an interactive 3-D scene. With `show=True` it is embedded directly in the notebook output, so you can orbit, zoom, and toggle between the surface-code and logical views inline. For a tall circuit, restrict the rendered cycle range along the z (time) axis with `start_cycle` / `end_cycle`.

In [ ]:
view_program(topo_program, show=True);

> **Tip:** The viewer supports orbit, zoom, pan, toggling between surface-code and logical views, camera presets, cycle-range filtering, and saving to an HTML file. See the [Visualization user guide](https://docs.classiq.io/user-guide/error-correction/visualization) for the full set of options.

## 7. Resources, fidelity, and runtime

The routed `TopologicalProgram` carries everything we need for a combined summary. For each code distance we report the **physical-qubit count** (a distance-`d` surface-code patch uses a `(d + 1) x (d + 1)` qubit footprint, times the patch-grid footprint) and the estimated **total logical error** and **fidelity** from `estimate_total_errors`. Larger distances suppress the error further, at the cost of more physical qubits. We also record how long the FT-engine steps took.

See the [Total Error user guide](https://docs.classiq.io/user-guide/error-correction/total-error) for more on how the error estimate is computed.

In [ ]:
def physical_qubits(code_distance: int) -> int:
    return num_patches * 2 * (code_distance + 1) ** 2


code_distances = [11, 13, 15]

total_errors = estimate_total_errors(topo_program, logical_noise, code_distances)

summary = pd.DataFrame(
    {
        "code_distance": code_distances,
        "physical_qubits": [physical_qubits(d) for d in code_distances],
        "total_error": [total_errors[d] for d in code_distances],
        "fidelity": [1.0 - total_errors[d] for d in code_distances],
    }
)
summary

## Cleanup and next steps

If you initialized your own logical noise, remove it when you no longer need it. (Shared public models such as `public/depolarize_1e-3` cannot be removed this way.)

From here you can scale up the circuit, sweep `clifford_t_approximation_error` to trade T-count against accuracy, or compare code distances to pick the smallest one that meets your target fidelity. For the full API details see the [SDK reference](https://docs.classiq.io/sdk-reference/error-correction); for conceptual background and more examples see the [Error Correction user guide](https://docs.classiq.io/user-guide/error-correction).

In [ ]:
if INITIALIZE_OWN_NOISE:
    remove_logical_noise(NOISE_NAME)